# Investigate the similarities between the templates for CUT1 (bdt_all > 0.75)

### Setting up the data and tools

In [1]:
# Importing the notebook with common setup 
%run 'bdt-all-common.ipynb'

Welcome to JupyROOT 6.24/06


Invoke (root_datasets, pandas_datasets) = load_data(inclmc_type="23903000") to load datasets
Invoke (root_datasets, pandas_datasets) = load_data(inclmc_type="23903003") to load dataset for double charm
Invoke  df_signal_23903000 = load_signal_from_inclMC() to load signal from 23903000 Inclusive MC
or
Invoke  df_signal = load_signal_all()
Invoke  df_background = load_background_category(category)
Loading BDTs from /media/lben/Samsung_T7/data/eos/lhcb/wg/semileptonic/RDsHad/AP/final/train_bdt/output2


In [2]:
import seaborn as sn
plt.rcParams["figure.figsize"] = (15,4)

## Applying  CUT1 i.e. BDT_all > 0.75

In [3]:
bdt_cutval=0.75

In [4]:
dfcut = df.query(f"bdt_all > {bdt_cutval}")
c = mygroupby(dfcut, 'category')
c['name'] = c.apply(lambda row: categories[f"{row['category']:.4g}"], axis=1)
c

,category,count,Percentage,cumulative %,name
0,19,3948,21.547866,21.547866,Xc_signal_Ypis_displaced_fromB0_fromDp
1,24,3305,18.038424,39.586290,Xc_signal_Ypis_displaced_fromBs_fromTau
2,22,1980,10.806680,50.392970,Xc_signal_Ypis_displaced_fromBs_fromDp
3,0,1641,8.956446,59.349416,Xc_background
4,14,1590,8.678092,68.027508,Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB
5,20,1299,7.089837,75.117345,Xc_signal_Ypis_displaced_fromBp_fromD0
6,18,1130,6.167449,81.284794,Xc_signal_Ypis_displaced_fromBs_fromDs
7,23,869,4.742932,86.027726,Xc_signal_Ypis_displaced_fromBp_fromDp
8,7,710,3.875123,89.902849,Xc_signal_Ypis_nomatch_doubleCharm
9,25,322,1.757450,91.660299,Xc_signal_Ypis_displaced_fromB0_fromD0


There are 17 categories represented by more than 100 candidates

## Using Kolmogorov-Smirnov test to see which distributions are similar

In [5]:
shown_number=17
sq2_2, stauY_2, sbdt_all = similar_categories(dfcut, shown_number)

TypeError: ks_2samp() got an unexpected keyword argument 'method'

In [ ]:
# Prepare some data to display histogramsand add to display tools
shown_categs = list(c.head(shown_number)['category'])
datasets = { f"{c}":  dfcut.query(f"category == {c}") for c in shown_categs }
datasets["others"] =  dfcut.query(f"category not in {shown_categs}")
datasets_names = { f"{c}": f"{c}:" + categories[f"{c}"] for c in shown_categs}
datasets_names["others"] = "others"

def plot_templates(datasets, datasets_names, myvar):
    htype = 'stepfilled'
    a = 0.30
    plt.hist([ d[myvar] for d in datasets ], bins=40, label=datasets_names, density=True, histtype=htype, alpha=a)
    if myvar == "tauY_2":
        plt.xlim([0, 0.004])
    plt.title(myvar)
    plt.legend()

def plot_templates_categs(mycategs, myvar):
    plot_templates([ datasets[d] for d in mycategs ], [ datasets_names[d] for d in mycategs ], myvar)

## Checking q2_2 distributions

In [ ]:
plt.figure()
sn.heatmap(sq2_2)
plt.title("Similarity of q2_2 distributions for the categories");

In [ ]:
from pprint import pprint
clusters = find_and_merge_clusters(sq2_2, 0.1)
pprint(clusters)

In [ ]:
plt.rcParams["figure.figsize"] = (15,4)
for c in clusters:
    plt.figure()
    plot_templates_categs(c, 'q2_2')

## Checking tauY_2 distributions

In [ ]:
plt.figure()
sn.heatmap(stauY_2)
plt.title("Similarity of tauY_2 distributions for the categories");

In [ ]:
from pprint import pprint
clusters = find_and_merge_clusters(stauY_2, 0.1)
pprint(clusters)

In [ ]:
for c in clusters:
    plt.figure()
    plot_templates_categs(c, 'tauY_2')

## Checking bdt_all distributions

In [ ]:
plt.figure()
sn.heatmap(sbdt_all)
plt.title("Similarity of bdt_all distributions for the categories");

In [ ]:
from pprint import pprint
clusters = find_and_merge_clusters(sbdt_all, 0.5)
pprint(clusters)

In [ ]:
for c in clusters:
    plt.figure()
    plot_templates_categs(c, 'bdt_all')